In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [17]:
# Fazendo a leitura dos dados
abt_train = pd.read_parquet(r'..\data\processed\bank_train_dataprep.parquet')

if 'y' in abt_train.columns:
    abt_target = abt_train['y']
    abt_train = abt_train.drop(columns=['y'])
else:
    X_fs = abt_train.copy()

# Configuração para visualizar todas as variávels
pd.set_option('display.max_columns', None)

# VarianceThreshold

In [18]:
# Definindo o limiar
selector = VarianceThreshold(threshold=0.01)

selector.fit_transform(abt_train)

# Colunas selecionadas
selected_features = abt_train.columns[selector.get_support()]

# Colunas descartadas
discarted_features = abt_train.columns[~selector.get_support()]
print(f'As colunas descartadas são: ', discarted_features)

# Armazenando somente as variáveis selecionadas
abt_01 = abt_train[selected_features]

print(f'Estrutura da base original: {abt_train.shape[1]}')
print(f'Estrutura da base com as variáveis selecionadas: {abt_01.shape[1]}')

As colunas descartadas são:  Index(['job_unknown', 'marital_unknown', 'default_yes', 'month_dec'], dtype='str')
Estrutura da base original: 53
Estrutura da base com as variáveis selecionadas: 49


# Correlação de Pearson

In [ ]:
# Calcular a correlação de Pearson
matriz_corr = abt_01.corr(method='pearson')

# Transformando em uma tabela empilhada de pares
pares_corr = matriz_corr.unstack()

# Criando um DataFrame para o resultado
df_top_corr = pd.DataFrame(pares_corr, columns=['Correlacao']).reset_index()
df_top_corr.columns = ['Variavel_1', 'Variavel_2', 'Correlacao']

# Remover a correlação da variável com ela mesma
df_top_corr = df_top_corr[df_top_corr['Variavel_1'] != df_top_corr['Variavel_2']]

# Remover o espelhamento da variável (ex: 'age' x 'euribor3m' e 'euribor3m' x 'age')
df_top_corr['Chave_Par'] = df_top_corr.apply(
    lambda row: '-'.join(sorted([row['Variavel_1'], row['Variavel_2']])), axis=1
)

df_top_corr = df_top_corr.drop_duplicates(subset=['Chave_Par']).drop(columns=['Chave_Par'])

# Valor absoluto para o ranking (ignora o sinal negativo)
df_top_corr['Correlacao_Abs'] = df_top_corr['Correlacao'].abs()

# Ordena pelas correlações mais fortes
df_top_corr = df_top_corr.sort_values(by='Correlacao_Abs', ascending=False)

print('== TOP MAIORES CORRELAÇÕES DE PEARSON ==')
display(df_top_corr.head(30))

== TOP MAIORES CORRELAÇÕES DE PEARSONN ==


,Variavel_1,Variavel_2,Correlacao,Correlacao_Abs
751,month_int,months_since_campaign,-1.000000,1.000000
1502,housing_unknown,loan_unknown,1.000000,1.000000
111,campaign,ratio_campaign_previous,0.997176,0.997176
110,campaign,total_contacts,0.983422,0.983422
253,emp.var.rate,euribor3m,0.972192,0.972192
601,total_contacts,ratio_campaign_previous,0.972062,0.972062
259,emp.var.rate,economic_stress_index,0.957737,0.957737
538,flg_contacted_prev,poutcome_success,0.948152,0.948152
401,euribor3m,nr.employed,0.944874,0.944874
406,euribor3m,economic_stress_index,0.923727,0.923727


In [ ]:
# Excluindo as variáveis contínuas que estão com extrema correlação
var_correlacionadas = ['month_int', 'total_contacts', 'ratio_campaign_previous']
abt_02 = abt_01.drop(columns=var_correlacionadas, errors='ignore')

# VIF - Variance Inflation Factor

In [ ]:
def remover_multicolinearidade_vif(df, limite_vif=10.0, proteger_binarias=True):
    """
    Cálcula o VIF iterativamente e remove automaticamente as variáveis
    contínuas que ultrapassa o limite de multicolineariedade
    
    Params:
    - df: DataFrame do Pandas contendo as features.
    - limite_vif: Float. Limite aceitável do VIF (padrão 10.0)
    - proteger_binarias: se True, ignora colunas <= 2 valores únicos (One-Hot Encoding)

    return:
    df_final: DataFrame com as colunas sobreviventes (incluindo as binárias que foram protegidas)
    colunas_removidas: Lista contendo o nome das variáveis contínuas excluidas.
    """

    # Filtras as colunas que passarão pelo VIF
    if proteger_binarias:
        # Pega apenas as colunas com nunique > 2
        cols_para_vif = [col for col in df.columns if df[col].nunique() > 2]
        print(f'Protegendo colunas binárias. O VIF analisará {len(cols_para_vif)} variáveis')
    else:
        cols_para_vif = list(df.columns)

    df_vif = df[cols_para_vif].copy()
    colunas_removidas = []

    while True:
        vif_data = pd.DataFrame()
        vif_data['Feature'] = df_vif.columns

        # Cálcula o VIF Dinamicamente
        vif_data['VIF'] = [
            variance_inflation_factor(df_vif.values, i)
            for i in range(df_vif.shape[1])
        ]

        # Encontra o maior valor
        max_vif = vif_data['VIF'].max()
        feature_max_vif = vif_data.loc[vif_data['VIF'] == max_vif, 'Feature'].values[0]

        # Corta se for maior ou igual que o limite
        if max_vif >= limite_vif:
            print(f'Removendo: {feature_max_vif}; (VIF: {max_vif:.2f})')
            df_vif = df_vif.drop(columns=[feature_max_vif])
            colunas_removidas.append(feature_max_vif)
        else:
            break

    # Reconstrói o DataFrame final (Variáveis Binárias + Contínuas que ficaram)
    colunas_finais = [c for c in df.columns if c not in colunas_removidas]
    df_final = df[colunas_finais]

    print('== Faxina Concluída!! ==')
    print(f'Colunas descartadas por multicolineariedade: {colunas_removidas}')
    print(f'Novo tamanho total da base: {df_final.shape[1]} colunas')

    return df_final, colunas_removidas

In [22]:
abt_03, colunas_excluidas = remover_multicolinearidade_vif(abt_02, 10, True)

Protegendo colunas binárias. O VIF analisará 13
Removendo: euribor3m; (VIF: 116.64)
Removendo: emp.var.rate; (VIF: 34.94)
Removendo: economic_stress_index; (VIF: 12.67)
== Faxina Concluída!! ==
Colunas descartadas por multicolineariedade: ['euribor3m', 'emp.var.rate', 'economic_stress_index']
Novo tamanho total da base: 43 colunas


# Correlação de Pearson 2.0

In [ ]:
def remove_alta_correlacao(df, threshold=0.8):
    

,age,education,campaign,pdays,previous,cons.price.idx,cons.conf.idx,nr.employed,flg_contacted_prev,total_credits_active,months_since_campaign,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,marital_married,marital_single,default_unknown,housing_unknown,housing_yes,loan_unknown,loan_yes,contact_telephone,month_aug,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
7309,0.800000,1.0,0.0,0.0,0.0,0.266594,0.857143,0.000000,0.0,-1.0,0.333333,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4169,0.866667,5.0,1.5,0.0,0.0,0.266594,0.857143,0.000000,0.0,-1.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
35810,0.533333,5.0,0.0,0.0,0.0,-0.931447,-0.698413,-0.712403,0.0,0.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
9668,0.533333,6.0,-0.5,0.0,0.0,0.779108,0.000000,0.287597,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
34389,-0.866667,4.0,1.5,0.0,0.0,-0.931447,-0.698413,-0.712403,0.0,-1.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
